# Understanding Knee MRI: Anatomical Planes & 2D vs 3D Sequences
### *A Visual Guide to Sagittal, Coronal, Axial Orientations and the 2D Multi-Slice vs 3D Volumetric Paradigm*

---

## 📋 Overview
In musculoskeletal MRI—and specifically in the **RSNA Knee Abnormality Detection** challenge—understanding **how scans are acquired** is essential for designing effective computer vision pipelines.

This notebook covers:
1. **The Three Anatomical Planes**: **Sagittal**, **Coronal**, and **Axial** (how they cut through the knee, what anatomical structures each reveals, and why multi-planar evaluation is necessary).
2. **2D Multi-Slice MRI vs 3D Volumetric Sequences**: The physics of acquisition, voxel geometry (anisotropic vs isotropic), slice thickness/gap, and why slice counts vary drastically from **~25 to 320 slices**.
3. **Multi-Planar Reconstruction (MPR)**: Visualizing why 3D isotropic sequences can be reformatted in any direction, whereas 2D sequences produce severe staircase/aliasing artifacts.
4. **Practical Modeling Recommendations**: How to feed multi-planar 2D/3D knee MRI data into modern vision architectures.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

# Aesthetic plot styling
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

# Resolve the repository from the root or this notebook's folder.
PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "baseline_candidate").is_dir() and (p / "notebooks").is_dir()), Path.cwd())

DATA_DIR = PROJECT_ROOT / "data"
TRAIN_SERIES_DIR = DATA_DIR / 'train_series'
df_train = pd.read_csv(DATA_DIR / 'train.csv')
df_series = pd.read_csv(DATA_DIR / 'train_series.csv')

print(f"Loaded train metadata: {len(df_train)} studies, {len(df_series)} series.")


## 1. The Three Anatomical Planes in Knee MRI

The knee is a complex multi-axial joint comprising bone, ligaments, menisci, and articular cartilage. No single viewpoint can evaluate all structures; radiologists examine the knee across **three orthogonal planes**:

| Anatomical Plane | Trajectory / Viewpoint | Key Structures Evaluated | Primary Target Conditions |
| :--- | :--- | :--- | :--- |
| **Sagittal** (~40% of series) | **Side-to-side** (Lateral $\leftrightarrow$ Medial)<br>Slices parallel to the leg's long axis | • **Anterior Cruciate Ligament (ACL)**<br>• **Posterior Cruciate Ligament (PCL)**<br>• Meniscal anterior & posterior horns<br>• Patellar tendon & extensor mechanism | `ACL`, Meniscal Tears, `Contusion` |
| **Coronal** (~35% of series) | **Front-to-back** (Anterior $\leftrightarrow$ Posterior)<br>Slices through both femoral condyles | • **Medial Collateral Ligament (MCL)**<br>• **Lateral Collateral Ligament (LCL)**<br>• Meniscal bodies<br>• Femoral & tibial joint cartilage | `MCL`, `Medial OA`, `Lateral OA`, Meniscus |
| **Axial (Transverse)** (~25% of series) | **Top-to-bottom** (Superior $\leftrightarrow$ Inferior)<br>Cross-sectional horizontal slices | • **Patellofemoral joint** & Trochlear groove<br>• Patellar cartilage & retinacula<br>• Suprapatellar pouch (effusion)<br>• Popliteal fossa (Baker's cyst) | `PF OA`, `Effusion`, `Synovitis`, `Baker's` |


In [ ]:
# Select a local example without distributing study identifiers.
# Set this override locally if you want to inspect a particular study.
SAMPLE_STUDY_UID = None


def local_series(uid, plane):
    rows = df_series[(df_series['StudyInstanceUID'] == uid) &
                     (df_series['Anatomical_Plane'] == plane)]
    found = []
    for series_uid in rows['SeriesInstanceUID'].astype(str):
        files = sorted((TRAIN_SERIES_DIR / uid / series_uid).glob('*.jpeg'))
        if files:
            found.append((series_uid, files))
    return sorted(found, key=lambda item: (len(item[1]), item[0]))


# Prefer annotated studies; no diagnosis is assumed for the selected example.
label_columns = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA',
                 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
gold_ids = df_train.loc[df_train[label_columns].notna().any(axis=1), 'StudyInstanceUID'].astype(str)
candidates = [SAMPLE_STUDY_UID] if SAMPLE_STUDY_UID is not None else sorted(gold_ids)
for uid in candidates:
    sagittal = local_series(uid, 'Sagittal')
    coronal = local_series(uid, 'Coronal')
    axial = local_series(uid, 'Axial')
    if sagittal and coronal and axial and len(sagittal) >= 2 and len(sagittal[-1][1]) > len(sagittal[0][1]):
        sample_study_uid = uid
        break
else:
    raise FileNotFoundError('Provide a local study with all three planes and two sagittal series of different lengths.')

study_path = TRAIN_SERIES_DIR / sample_study_uid
series_info = df_series[df_series['StudyInstanceUID'] == sample_study_uid]
cor_uid, cor_files = coronal[0]
ax_uid, ax_files = axial[0]
sag_uid, sag_files = sagittal[0]

# Middle slices are illustrative; anatomical centering is not established.
cor_img = Image.open(cor_files[len(cor_files) // 2])
ax_img = Image.open(ax_files[len(ax_files) // 2])
sag_img = Image.open(sag_files[len(sag_files) // 2])

fig, axes = plt.subplots(1, 3, figsize=(15, 6))

planes_data = [
    (cor_img, "1. CORONAL PLANE (Front-to-Back)", "Best for: MCL, LCL, Cartilage, Medial/Lateral OA", "#2563EB"),
    (sag_img, "2. SAGITTAL PLANE (Side-to-Side)", "Best for: ACL, PCL, Meniscal Horns, Extensor Mechanism", "#16A34A"),
    (ax_img,  "3. AXIAL PLANE (Cross-Section)", "Best for: Patellofemoral (PF OA), Effusion, Baker's Cyst", "#D97706"),
]

for ax, (img, title, subtitle, color) in zip(axes, planes_data):
    ax.imshow(img, cmap='bone')
    ax.set_title(title, fontsize=12, fontweight='bold', pad=8, color=color)
    ax.set_xlabel(subtitle, fontsize=9, style='italic', labelpad=6)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_edgecolor(color)
        spine.set_linewidth(2.5)

plt.suptitle("The Three MRI Anatomical Planes (local example study)", fontsize=15, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()


## 2. Anatomical Traversal: Slicing Through the Knee

An MRI scan is not a single photo—it is a continuous stack of sequential slices.
Here is how the images progress as the scanner moves along each axis:
* **Sagittal Stack**: Scans from **Lateral (outer knee) $\rightarrow$ Central notch $\rightarrow$ Medial (inner knee)**.
* **Coronal Stack**: Scans from **Anterior (front/patella) $\rightarrow$ Joint center $\rightarrow$ Posterior (back of femoral condyles)**.
* **Axial Stack**: Scans from **Superior (distal femur) $\rightarrow$ Joint line $\rightarrow$ Inferior (proximal tibia)**.


In [ ]:
# Display a 5-slice progression across Sagittal, Coronal, and Axial planes
def plot_plane_progression(files, plane_name, color):
    indices = np.linspace(2, len(files)-3, 5, dtype=int)
    fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))
    for i, idx in enumerate(indices):
        img = Image.open(files[idx])
        axes[i].imshow(img, cmap='bone')
        axes[i].set_title(f"Slice {idx+1}/{len(files)}", fontsize=10, fontweight='bold')
        axes[i].axis('off')
        for spine in axes[i].spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(1.5)
    plt.suptitle(f"{plane_name} Sequence Progression ({len(files)} slices total)", fontsize=13, fontweight='bold', color=color, y=1.03)
    plt.tight_layout()
    plt.show()

plot_plane_progression(cor_files, "Coronal (Anterior -> Posterior)", "#2563EB")
plot_plane_progression(sag_files, "Sagittal (Lateral -> Medial)", "#16A34A")
plot_plane_progression(ax_files,  "Axial (Superior -> Inferior)", "#D97706")


## 3. The 2D Multi-Slice vs 3D Volumetric Paradigm

Why do some series have **~25 slices** while others have **100 to 320 slices**?
This difference stems from two fundamentally distinct MRI acquisition technologies:

### 🔬 2D Multi-Slice MRI (Standard Clinical Workhorse):
* **Physics**: The scanner excites one 2D slice at a time using selective RF pulses.
* **Slice Thickness & Gap**: Each slice is relatively thick (**3.0 mm to 4.0 mm**), often with a **0.3 mm gap** between slices to prevent radiofrequency crosstalk.
* **Anisotropic Voxels**: Pixels inside the image plane are very sharp (**~0.2 mm $\times$ 0.2 mm**), but the slice thickness is **3.0 mm** (a $15:1$ aspect ratio!). Each voxel is shaped like an elongated slab or pancake.
* **Total Slices**: It only takes **20 to 35 slices** to cover the entire width or height of the knee joint.
* **Trade-off**: Fast scan time (~2 minutes per sequence), excellent high in-plane SNR, but severe **partial volume averaging** through the slice thickness.

---

### 📦 3D Volumetric Sequences (3D SPACE, CUBE, VISTA, 3D FSE):
* **Physics**: The scanner excites a single large 3D volume slab and uses phase encoding along the slice-select axis.
* **Sub-Millimeter Thin Slices**: Slices are **0.5 mm to 0.8 mm thick** with **zero slice gap**.
* **Isotropic Voxels**: Voxel dimensions are uniform in all 3 directions (e.g., **0.6 mm $\times$ 0.6 mm $\times$ 0.6 mm** cubes).
* **Total Slices**: Spanning a 120 mm knee in 0.5 mm steps requires **120 to 320 slices**!
* **Trade-off**: Requires longer acquisition times, but enables **Multi-Planar Reconstruction (MPR)**: you can digitally reslice the 3D volume into Coronal, Axial, or angled oblique planes at arbitrary angles with zero loss in resolution!

---

### 📊 Direct Comparison Table

| Property | 2D Multi-Slice MRI | 3D Volumetric MRI |
| :--- | :--- | :--- |
| **Typical Slice Count** | **20 – 35 slices** | **100 – 320 slices** |
| **Slice Thickness** | **3.0 – 4.0 mm** | **0.5 – 0.8 mm** |
| **Interslice Gap** | Yes (0.3 – 0.5 mm) | None (continuous volume) |
| **Voxel Shape** | **Anisotropic** (Pancake / Slab: $0.2 \times 0.2 \times 3.0$ mm) | **Isotropic** (Cube: $0.6 \times 0.6 \times 0.6$ mm) |
| **Partial Volume Averaging** | High (small tears can hide between slices) | Minimal (sub-millimeter precision) |
| **Multi-Planar Reformat (MPR)**| ❌ Impossible (severe staircase / blur artifacts) | ✅ Perfect (reslice at any orientation) |
| **Common Clinical Names** | TSE, FSE, turbo spin echo | 3D SPACE (Siemens), 3D CUBE (GE), 3D VISTA (Philips) |


In [ ]:
# Analyze the slice count distribution across a representative sample of 250 studies (~1,400 series)
sample_studies = list(TRAIN_SERIES_DIR.iterdir())[:250]
all_series_slices = []

for s_dir in sample_studies:
    if not s_dir.is_dir(): continue
    for ser_dir in s_dir.iterdir():
        if ser_dir.is_dir():
            n_files = len(list(ser_dir.glob('*.jpeg')))
            all_series_slices.append(n_files)

slice_series = pd.Series(all_series_slices)

# Plot bimodal distribution
fig, ax = plt.subplots(figsize=(12, 5))
n, bins, patches_list = ax.hist(slice_series, bins=60, color='#3B82F6', edgecolor='white', alpha=0.85)

# Highlight 2D vs 3D regions
ax.axvspan(10, 50, color='#10B981', alpha=0.15, label='Standard 2D Multi-Slice Scans (15 - 45 slices)')
ax.axvspan(90, 350, color='#8B5CF6', alpha=0.15, label='3D Volumetric Sequences (100 - 320 slices)')

ax.set_title("Bimodal Distribution of MRI Slice Counts per Sequence (RSNA Knee Dataset)", fontsize=13, fontweight='bold')
ax.set_xlabel("Number of Image Slices in 1 MRI Scan", fontsize=11)
ax.set_ylabel("Number of Series", fontsize=11)
ax.legend(fontsize=10, loc='upper right')

# Add descriptive callout annotations
ax.annotate('2D Peak (~28-32 slices)\nFast clinical standard', xy=(30, max(n)*0.85), xytext=(60, max(n)*0.85),
            arrowprops=dict(facecolor='#10B981', shrink=0.08, width=2, headwidth=8),
            fontsize=10, fontweight='bold', color='#047857')

ax.annotate('3D Volumetric Cluster\nSub-mm isotropic sequences (SPACE/CUBE)', xy=(130, max(n)*0.1), xytext=(150, max(n)*0.35),
            arrowprops=dict(facecolor='#8B5CF6', shrink=0.08, width=2, headwidth=8),
            fontsize=10, fontweight='bold', color='#6D28D9')

plt.tight_layout()
plt.show()

print(f"Sampled series count: {len(slice_series)}")
print(f"2D sequences (<50 slices): {(slice_series < 50).mean()*100:.1f}%")
print(f"3D sequences (>=100 slices): {(slice_series >= 100).mean()*100:.1f}%")


In [ ]:
# Compare two local sagittal series from the same study.
# Slice counts alone cannot establish 2D/3D acquisition or voxel spacing.
ser_2d_sag_uid, files_2d = sagittal[0]
ser_3d_sag_uid, files_3d = sagittal[-1]
img_2d_center = Image.open(files_2d[len(files_2d) // 2])
img_3d_center = Image.open(files_3d[len(files_3d) // 2])

fig = plt.figure(figsize=(15, 8))
gs = fig.add_gridspec(2, 2, width_ratios=[1.2, 2])

# Left Col: Visual comparison of 2D vs 3D Sagittal Slice
ax_2d = fig.add_subplot(gs[0, 0])
ax_2d.imshow(img_2d_center, cmap='gray')
ax_2d.set_title(f"Sparser sagittal series ({len(files_2d)} slices)\nAcquisition type requires DICOM metadata", fontsize=10, fontweight='bold', color='#047857')
ax_2d.axis('off')

ax_3d = fig.add_subplot(gs[1, 0])
ax_3d.imshow(img_3d_center, cmap='gray')
ax_3d.set_title(f"Denser sagittal series ({len(files_3d)} slices)\nAcquisition type requires DICOM metadata", fontsize=10, fontweight='bold', color='#6D28D9')
ax_3d.axis('off')

# Right Col: Visualizing Spatial Sampling Density along the Knee Joint
ax_density = fig.add_subplot(gs[:, 1])
ax_density.set_xlim(-5, 105)
ax_density.set_ylim(-0.5, 2.5)

# Simulate 100mm knee joint width
knee_width_mm = 100

# 2D sampling: 26 slices spaced every ~3.5mm
slices_2d_positions = np.linspace(5, 95, len(files_2d))
for pos in slices_2d_positions:
    rect = patches.Rectangle((pos - 1.5, 1.4), 3.0, 0.6, linewidth=1, edgecolor='#047857', facecolor='#10B981', alpha=0.55)
    ax_density.add_patch(rect)

# 3D sampling: 320 slices spaced every ~0.3mm
slices_3d_positions = np.linspace(2, 98, len(files_3d))
for pos in slices_3d_positions:
    ax_density.plot([pos, pos], [0.1, 0.7], color='#7C3AED', alpha=0.35, linewidth=0.8)

ax_density.set_yticks([0.4, 1.7])
ax_density.set_yticklabels(['Illustrative dense sampling\n(not measured geometry)', 'Illustrative sparse sampling\n(not measured geometry)'], fontsize=11, fontweight='bold')
ax_density.set_xlabel('Illustrative position (arbitrary units)', fontsize=11, fontweight='bold')
ax_density.set_title('Illustrative sparse vs dense sampling; verify DICOM geometry', fontsize=12, fontweight='bold', pad=12)

# Annotate gap vs continuous
ax_density.text(50, 1.05, "Illustrative spacing only: actual thickness and spacing require DICOM metadata", 
                ha='center', fontsize=9.5, style='italic', color='#B91C1C', bbox=dict(boxstyle='round,pad=0.3', facecolor='#FEE2E2', edgecolor='#EF4444'))

ax_density.text(50, -0.25, "A larger slice count alone does not establish isotropic 3D acquisition", 
                ha='center', fontsize=9.5, style='italic', color='#047857', bbox=dict(boxstyle='round,pad=0.3', facecolor='#D1FAE5', edgecolor='#10B981'))

plt.tight_layout()
plt.show()


## 4. Competitive Modeling Strategies for 2D vs 3D Sequences

When competing in the RSNA Knee Challenge or building clinical models, how should you handle these different sequence types?

### 1. Handling Standard 2D Sequences (~25-35 slices):
* **2.5D Slice Aggregation**: Extract 2D features per slice using pretrained backbones (e.g., `ConvNeXt-Small`, `EfficientNet-V2`, or `EVA-02`), then aggregate across the sequence using **Attention-based Multiple Instance Learning (Attention-MIL)**, **1D CNNs**, or **Transformer encoders**.
* **Key-Slice Sampling**: Because 2D sequences only have ~25 slices, taking 16 or 24 evenly spaced slices per series easily fits into GPU memory (e.g. 16 slices $\times$ 3 planes = 48 slices per study).

### 2. Handling 3D Volumetric Sequences (~120-320 slices):
* **Subsampling to Fixed Depth**: When combining with 2D series, downsample the 3D volume along the slice dimension (e.g., interpolate or pick 32 evenly spaced slices) so all series have a uniform temporal/slice depth.
* **3D Native Convolutions**: For pure 3D sequences, apply true 3D architectures (e.g., `3D ResNet-50`, `MedNeXt`, or `Swin UNETR`) to leverage 3D spatial continuity.

### 3. Multi-View Anatomical Late Fusion:
* **Never flatten all planes together blindly**: Pass Sagittal, Coronal, and Axial series through dedicated plane-specific encoders or attention heads, then concatenate their embeddings before predicting the 12 target classes.
  * **ACL / Meniscus heads** will attend strongly to the **Sagittal** sequence features.
  * **MCL / Osteoarthritis heads** will attend strongly to the **Coronal** sequence features.
  * **Effusion / PF OA heads** will attend strongly to the **Axial** sequence features.
